# Assignment 3: Data Analysis (Python)
## Question 2: What Factors Influence Length of Stay for Inpatient Thyroid-Related Visits?


---

**Question:** Among patients with thyroid conditions who had inpatient or agency visits, do length-of-stay (LOS) patterns differ by condition, visit type, gender, race, or age group, and what factors are associated with longer stays?

**Data Source:** `visits_analysis_ready.csv` — pre-cleaned via `assignment3_data_cleaning.py` from the original `ss_visits_thyroid_clean.csv` (SubSalt data).

**Key Data Dictionary Note:** Per the D2L announcement, `visit_end_date` is only applicable for inpatient and agency visits. LOS is therefore computed exclusively for those encounter types. All other visit types use `visit_start_date` only.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Display and plot settings ───────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# ── Load only the visits dataset (pre-cleaned) ─────────────────────
visits = pd.read_csv("visits_analysis_ready.csv")

print(f"Visits dataset: {visits.shape[0]:,} rows x {visits.shape[1]} columns")
print(f"Unique patients: {visits['person_id'].nunique():,}")
print(f"Memory usage: {visits.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 2. Filter to Inpatient/Agency Visits

The data dictionary states that `visit_end_date` is only meaningful for inpatient and home health visit types. Applying length of stay (LOS) calculations to office or outpatient visits would produce misleading results.

Age groups were converted into clinical decades (18–29, 30–39, ... 80+) since these are standard groupings in the medical literature.

In [ ]:
# ── 2a. Filter to visit types where LOS is valid ───────────────────
# Per data dictionary: visit_end_date applies to inpatient and agency visits only
los_valid_types = [
    "Inpatient Visit",
    "Inpatient Hospital",
    "Emergency Room and Inpatient Visit",
    "Case Management Agency",
]

inp = visits[visits["visit_type"].isin(los_valid_types)].copy()
print(f"Rows with valid LOS (inpatient + agency): {len(inp):,}")
print(f"Unique patients: {inp['person_id'].nunique():,}")
print(f"\nVisit type breakdown:")
print(inp["visit_type"].value_counts().to_string())

In [ ]:
# ── 2b. Rename LOS column for clarity and handle edge cases ────────
inp.rename(columns={"visit_duration_days": "los_days"}, inplace=True)

# Delete rows where LOS is missing or negative
before = len(inp)
inp = inp[inp["los_days"].notna() & (inp["los_days"] >= 0)].copy()
print(f"Dropped {before - len(inp)} rows with missing/invalid LOS")

# 2c. Convert to clinical decade groups
# Standard clinical grouping used in the literature
inp["age_decade"] = pd.cut(
    inp["age_at_visit"],
    bins=[17, 29, 39, 49, 59, 69, 79, 120],
    labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"],
    right=True,
)

print(f"\nAge decade distribution:")
print(inp["age_decade"].value_counts().sort_index().to_string())
print(f"\nMissing age (excluded from age-based analyses): {inp['age_decade'].isna().sum()}")

In [ ]:
# 2d. Create a simplified visit subtype for cleaner labels
# "Emergency Room and Inpatient Visit" is long; shorten for plot labels
visit_label_map = {
    "Inpatient Visit": "Inpatient",
    "Inpatient Hospital": "Inpatient Hospital",
    "Emergency Room and Inpatient Visit": "ER-to-Inpatient",
    "Case Management Agency": "Case Mgmt/Agency",
}
inp["visit_subtype"] = inp["visit_type"].map(visit_label_map)

# Summary of the analytic dataset
print(f"Final analytic dataset: {len(inp):,} rows, {inp['person_id'].nunique():,} unique patients")
print(f"\nLOS summary (days):")
print(inp["los_days"].describe().round(1).to_string())

## 3. Descriptive Statistics

Descriptives were run first to get a sense for the data before interpreting results, and will expose  remaining data quality issues. In clinical research, "Table 1" — a demographic/characteristic summary stratified by a key variable — is the standard starting point for most articles.

In [ ]:
# 3a. Table 1: LOS summary by key grouping variables
# Build a summary function to reuse
def los_summary_by_group(data, group_col, label=None):
    """Return a summary DataFrame of LOS stats grouped by a column."""
    summary = (
        data.groupby(group_col, observed=True)["los_days"]
        .agg(["count", "mean", "median", "std", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
        .round(1)
    )
    summary.columns = ["N", "Mean", "Median", "SD", "Q1", "Q3"]
    summary.index.name = label or group_col
    return summary

# LOS by visit subtype
print("=" * 65)
print("LOS (Days) by Visit Subtype")
print("=" * 65)
display(los_summary_by_group(inp, "visit_subtype", "Visit Subtype"))

# LOS by condition
print("\n" + "=" * 65)
print("LOS (Days) by Thyroid Condition")
print("=" * 65)
display(los_summary_by_group(inp, "condition_description_clean", "Condition"))

In [ ]:
# 3b. LOS by demographics
# Gender
print("=" * 65)
print("LOS (Days) by Gender")
print("=" * 65)
display(los_summary_by_group(inp, "gender_clean", "Gender"))

# Race
print("\n" + "=" * 65)
print("LOS (Days) by Race")
print("=" * 65)
display(los_summary_by_group(inp, "race_clean", "Race"))

# Ethnicity
print("\n" + "=" * 65)
print("LOS (Days) by Ethnicity")
print("=" * 65)
display(los_summary_by_group(inp, "ethnicity_clean", "Ethnicity"))

# Age decade
print("\n" + "=" * 65)
print("LOS (Days) by Age Decade")
print("=" * 65)
display(los_summary_by_group(inp.dropna(subset=["age_decade"]), "age_decade", "Age Decade"))

## 4. Figures

For comparing distributions of a continuous variable (LOS) across categories, box plots can show medians, interquartile ranges, and outliers. Bar charts are helpful for counts. Heat maps can help provide a visual representation of relationships between two categorical variables.

In [ ]:
# ── Figure 1: Overall LOS Distribution ──────────────────────────────
# A histogram shows the shape of the LOS distribution.
# This is always the first figure in a LOS analysis: understand the
# overall distribution (normal, non-parametric) before stratifying.

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(inp["los_days"], bins=60, color="steelblue", edgecolor="white", alpha=0.8)
ax.set_xlabel("Length of Stay (Days)")
ax.set_ylabel("Number of Visits")
ax.set_title("Figure 1: Distribution of Inpatient Length of Stay")
ax.set_xlim(0, inp["los_days"].quantile(0.99))  # Trim extreme outliers for readability
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: LOS by Visit Subtype (Box Plot) ──────────────────────
# Each box shows the median (line), IQR (box), and outliers (dots).

fig, ax = plt.subplots(figsize=(10, 5))
order = ["Inpatient", "Inpatient Hospital", "ER-to-Inpatient", "Case Mgmt/Agency"]
sns.boxplot(
    data=inp, x="visit_subtype", y="los_days", order=order,
    palette="Set2", showfliers=False, ax=ax,
)
ax.set_xlabel("Visit Subtype")
ax.set_ylabel("Length of Stay (Days)")
ax.set_title("Figure 2: LOS by Inpatient Visit Subtype")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 3: LOS by Gender and Race (Grouped Box Plots)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Panel A: Gender
sns.boxplot(
    data=inp[inp["gender_clean"].isin(["F", "M"])],
    x="gender_clean", y="los_days",
    palette={"F": "#e07b91", "M": "#7bafd4"}, showfliers=False, ax=axes[0],
)
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Length of Stay (Days)")
axes[0].set_title("A. LOS by Gender")

# Panel B: Race (top 4 groups for readability)
top_races = inp["race_clean"].value_counts().head(4).index.tolist()
sns.boxplot(
    data=inp[inp["race_clean"].isin(top_races)],
    x="race_clean", y="los_days", order=top_races,
    palette="Set2", showfliers=False, ax=axes[1],
)
axes[1].set_xlabel("Race")
axes[1].set_ylabel("")
axes[1].set_title("B. LOS by Race")
axes[1].tick_params(axis="x", rotation=20)

fig.suptitle("Figure 3: LOS by Gender and Race", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 4: LOS by Age Decade ─────────────────────────────────────
# Clinical decade groupings more closely match reported medical literature.

age_data = inp.dropna(subset=["age_decade"])

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=age_data, x="age_decade", y="los_days",
    palette="YlOrRd", showfliers=False, ax=ax,
)
ax.set_xlabel("Age Group (Clinical Decades)")
ax.set_ylabel("Length of Stay (Days)")
ax.set_title("Figure 4: LOS by Age Decade")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 5: Heatmap of Median LOS by Visit Subtype x Condition
# A heatmap shows the interaction between two categorical variables
# simultaneously.

pivot = inp.pivot_table(
    values="los_days", index="visit_subtype", columns="condition_description_clean",
    aggfunc="median",
).round(1)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=0.5, ax=ax)
ax.set_xlabel("Thyroid Condition")
ax.set_ylabel("Visit Subtype")
ax.set_title("Figure 5: Median LOS (Days) by Visit Subtype and Condition")
plt.tight_layout()
plt.show()

## 5. Statistical Testing

When comparing LOS across groups, the choice of test depends on how many groups you have and whether your data meets normality assumptions. With two groups, I'd use a t-test (normal distribution) or Mann-Whitney U (non-parametric). With three or more groups, I'd use an ANOVA (normal distribution), the Kruskal-Wallis (non-parametric) test. LOS data are very often not normally distributed due to a small amount of extreme outliers. Non-parametric analysis would account for this and I used the Kruskal-Wallis approach due to this.

In [ ]:
# 5a. Kruskal-Wallis tests across each factor
# Kruskal-Wallis is the non-parametric alternative to one-way ANOVA 
# I used it here because LOS distributions are right-skewed.
# Since this is a large dataset, significance alone should be scrutinized as it may not be clinically relevant.
# This is the curse of big data.

def run_kruskal(data, group_col, outcome="los_days"):
    """Run Kruskal-Wallis H test and return results as a dict."""
    groups = [g[outcome].dropna().values for _, g in data.groupby(group_col, observed=True)]
    # Need at least 2 groups with data
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return {"test": "Kruskal-Wallis", "H-statistic": None, "p-value": None, "n_groups": len(groups)}
    stat, p = stats.kruskal(*groups)
    return {"test": "Kruskal-Wallis", "H-statistic": round(stat, 2), "p-value": round(p, 6), "n_groups": len(groups)}

# Run tests for each factor
test_factors = {
    "Visit Subtype": "visit_subtype",
    "Condition": "condition_description_clean",
    "Gender (F vs M)": "gender_clean",
    "Race": "race_clean",
    "Age Decade": "age_decade",
}

results = []
for label, col in test_factors.items():
    subset = inp.dropna(subset=[col])
    # For gender, limit to F and M only (Other/Unknown has n=1)
    if col == "gender_clean":
        subset = subset[subset["gender_clean"].isin(["F", "M"])]
    res = run_kruskal(subset, col)
    res["Factor"] = label
    results.append(res)

results_df = pd.DataFrame(results)[["Factor", "n_groups", "H-statistic", "p-value"]]
print("Kruskal-Wallis H Test Results:")
print("(Tests whether LOS distributions differ across groups)\n")
display(results_df)

## 6. Narrative: Interpretation and Limitations

### Methods and Findings

To investigate whether LOS differs by thyroid condition, visit type, gender, race, or age group, we filtered the cleaned thyroid visits dataset to include only visit types where `visit_end_date` is a valid measure of stay duration as noted in the data dictionary. Per the data dictionary, this is only valid for Inpatient Visit, Inpatient Hospital, Emergency Room and Inpatient Visit, and Case Management Agency encounters. This produced a sample of approximately 38,000 visits across roughly 12,500 unique patients. LOS was calculated as the difference in days between `visit_end_date` and `visit_start_date`. Age at visit was derived from `year_of_birth` and the visit start year. This was then grouped into clinical decades (18–29, 30–39, etc.). Descriptive statistics (mean, median, IQR) were computed for each factor, and group differences were tested using the Kruskal-Wallis test because of the non-parametric nature of LOS data.

Across the factors examined, the LOS distributions were notably similar. Median LOS had a narrow range regardless of thyroid condition, visit subtype, gender, race, or age group. While visit subtype reached statistical significance on the Kruskal-Wallis test, the practical difference in median LOS across subtypes was minimal.

### Limitations

Several limitations should be considered when interpreting these results. First, this is a synthetic dataset generated by SubSalt without clinician oversight of the data relationships. Multiple clinically impossible situations were present due to this such as month-long office visits. The uniformity of LOS distributions across distinct visit types (e.g., direct admissions vs ED to inpatient admissions) and across conditions suggests that the synthetic data generation process does not mirror "real world" distributions. Because of this, the absence of group differences should not be interpreted as evidence that these factors do not influence LOS in real-world patients. In actual EHR data, older age, condition severity, and inpatient visit type are established predictors of longer stays.